In [226]:
import glob
import pandas as pd
import re
from pathlib import Path
import sys, os

from utils.data_processing import load_lexicon, add_clean_descriptors_column, remove_parentheses_text, DEFAULT_ADJECTIVES
import pandas as pd

# get adjectives from answers

In [227]:
all_desc = pd.read_excel("data/all_descriptors.xlsx")
all_desc_flavor = all_desc['flavor'].str.lower().str.strip().unique().tolist()

In [228]:
merged_df = pd.read_csv('data/merged_ai_descriptors.csv')
merged_df['ai_descriptors'] = merged_df['ai_descriptors'].apply(remove_parentheses_text)
merged_df['ai_descriptors']= merged_df['ai_descriptors'].str.replace(", ", ",")

merged_df_flavor = (
    merged_df["ai_descriptors"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

combined_flavor_descriptors = set(all_desc_flavor).union(set(merged_df_flavor))

In [229]:
# Extract flavor descriptor from RAG answers
descriptor_set = {
    d.strip().lower()
    for d in combined_flavor_descriptors
}

for r in records:
    answer = r.get("answer", "")

    # split model output
    tokens = [
        t.strip().lower()
        for t in answer.split(",")
        if t.strip()
    ]

    # keep only known flavor descriptors
    matched = [t for t in tokens if t in descriptor_set]

    r["matched_flavor_descriptors"] = list(set(matched))

In [230]:
def extract_flavor_descriptors(records, flavor_descriptors):
    """
    Extract known flavor descriptors from each record's 'answer' field.
    """
    descriptor_set = {d.strip().lower() for d in flavor_descriptors}

    new_records = []

    for r in records:
        answer = r.get("answer", "")

        tokens = [
            t.strip().lower()
            for t in answer.split(",")
            if t.strip()
        ]

        # preserve order + remove duplicates
        matched = list(dict.fromkeys(t for t in tokens if t in descriptor_set))

        new_r = r.copy()
        new_r["matched_flavor_descriptors"] = matched

        new_records.append(new_r)

    return new_records

# all_ing = extract_flavor_descriptors(records, combined_flavor_descriptors)


In [231]:
def extract_flavor_descriptors(records, flavor_descriptors):
    """
    Extract known flavor descriptors from each record's 'answer' field
    using messy/free-text matching (not CSV splitting).

    Returns a NEW list of dicts (does not mutate input records).
    """
    descriptor_list = sorted(
        {d.strip().lower() for d in flavor_descriptors if str(d).strip()},
        key=len,
        reverse=True,  # match longer phrases first (e.g., "slightly sweet")
    )

    pattern = re.compile(
        r"\b(?:%s)\b" % "|".join(map(re.escape, descriptor_list)),
        flags=re.I,
    )

    new_records = []
    for r in records:
        answer = (r.get("answer") or "").lower()

        found = pattern.findall(answer)
        matched = list(dict.fromkeys(f.lower() for f in found))  # unique, preserve order

        new_r = r.copy()
        new_r["matched_flavor_descriptors"] = matched
        new_records.append(new_r)

    return new_records


# all_ing = extract_flavor_descriptors(records, combined_flavor_descriptors)


# Load data

In [232]:
files = glob.glob("data/RAG/*.out")

In [233]:
# Native extraction 

path = files[0]
text = Path(path).read_text(errors="ignore")

# 1) Drop any preamble noise before the FIRST [PROMPT]
end = text.find("Running Flavor_DB ingredients\n")
text = text[:end]

chunks = text.split("[RAG]")

# Prompt regex (full prompt)
prompt_re = re.compile(
    r"(What does\s+.+?,\s+with the scientific name\s+.+?,\s+taste like\?\s+Give a comma-separated list of flavor adjectives\.)",
    flags=re.I | re.S
)

# Scientific name regex (captures ONLY the scientific name)
sci_name_re = re.compile(
    r"with the scientific name\s+([^,]+)",
    flags=re.I
)

records = []
current_prompt = None
current_sci_name = None

for chunk in chunks:
    # Extract prompt
    m = prompt_re.search(chunk)
    if m:
        current_prompt = m.group(1).strip()

        # Extract scientific name from prompt
        m_sci = sci_name_re.search(current_prompt)
        current_sci_name = m_sci.group(1).strip() if m_sci else None

    # Extract answer
    m2 = re.search(r"\[FINAL ANSWER\](.*)", chunk, flags=re.I | re.S)
    if m2 and current_prompt is not None:
        answer = m2.group(1).strip()

        records.append({
            "prompt": current_prompt,
            "scientific_name": current_sci_name,
            "answer": answer,
        })

        # Reset to avoid cross-pairing
        current_prompt = None
        current_sci_name = None

# Extract flavor descriptors using the defined function
native_ing = extract_flavor_descriptors(records, combined_flavor_descriptors)
native_ing[:1], len(native_ing)

([{'prompt': 'What does varech géant,, with the scientific name Macrocystis pyrifera, taste like? Give a comma-separated list of flavor adjectives.',
   'scientific_name': 'Macrocystis pyrifera',
   'answer': 'WeWe need to answer: "What does varech géant, with the scientific name Macrocystis pyrifera, taste like? Give a comma-separated list of flavor adjectives."\n\nWe have context about cooking, but not about Macrocystis pyrifera. The context is about barbecue, giant wolf\'s bladder (Calvatia gigantea). No mention of Macrocystis pyrifera. So we need to infer typical taste of giant kelp (Macrocystis pyrifera). Usually seaweed has briny, umami, salty, slightly sweet, marine, mineral, earthy, slightly bitter,',
   'matched_flavor_descriptors': ['seaweed',
    'briny',
    'umami',
    'salty',
    'slightly sweet',
    'marine',
    'mineral',
    'earthy',
    'slightly bitter']}],
 168)

In [234]:
# all ingredients (both native and flavorDB)

path = files[0]
text = Path(path).read_text(errors="ignore")

# 1) Drop any preamble noise before the FIRST [PROMPT]
end = text.find("Running Flavor_DB ingredients\n")
text = text[end:]


chunks = text.split("[PROMPT]")

# ---- Prompt patterns ----

# 1) With scientific name
prompt_sci_re = re.compile(
    r"(What does\s+.+?,\s+with the scientific name\s+.+?,\s+taste like\?\s+Give a comma-separated list of flavor adjectives\.)",
    flags=re.I | re.S
)

# 2) Common name only
prompt_common_re = re.compile(
    r"(What does\s+.+?,\s+taste like\?\s+Give a comma-separated list of flavor adjectives\.)",
    flags=re.I | re.S
)

# Extractors
sci_name_re = re.compile(
    r"with the scientific name\s+([^,]+)",
    flags=re.I
)

common_name_re = re.compile(
    r"What does\s+([^,]+)",
    flags=re.I
)

records = []
current_prompt = None
current_common = None
current_sci = None

for chunk in chunks:
    # --- Try scientific-name prompt first ---
    m = prompt_sci_re.search(chunk)
    if m:
        current_prompt = m.group(1).strip()

        m_sci = sci_name_re.search(current_prompt)
        current_sci = m_sci.group(1).strip() if m_sci else None

        m_common = common_name_re.search(current_prompt)
        current_common = m_common.group(1).strip() if m_common else None

    else:
        # --- Try common-name-only prompt ---
        m = prompt_common_re.search(chunk)
        if m:
            current_prompt = m.group(1).strip()
            current_sci = None

            m_common = common_name_re.search(current_prompt)
            current_common = m_common.group(1).strip() if m_common else None

    # --- Extract answer ---
    m2 = re.search(r"\[FINAL ANSWER\](.*)", chunk, flags=re.I | re.S)
    if m2 and current_prompt is not None:
        answer = m2.group(1).strip()

        records.append({
            "prompt": current_prompt,
            "common_name": current_common,
            "scientific_name": current_sci,
            "answer": answer,
        })

        # reset
        current_prompt = None
        current_common = None
        current_sci = None

# Extract flavor descriptors using the defined function
all_ing = extract_flavor_descriptors(records, combined_flavor_descriptors)
all_ing[:1], len(all_ing)


([{'prompt': 'What does Macrocystis pyrifera, taste like? Give a comma-separated list of flavor adjectives.',
   'common_name': 'Macrocystis pyrifera',
   'scientific_name': None,
   'answer': 'salty, briny, umami, slightly sweet, marine\n\n(Note: The answer is based on general culinary knowledge and may not be specific to Macrocystis pyrifera.)\n\nsalty, briny, umami, slightly sweet, marine\n\n(Note: The answer is based on general culinary\n\nsalty, briny, umami, slightly sweet, marine\n\n(Note: The answer is based on general culinary\n\nsalty, briny, umami, slightly\n\nsalty, briny, umami, slightly sweet, marine\n\n(Note: The answer\n\nsalty\n\nsalty\n\nsalty\n\nsal',
   'matched_flavor_descriptors': ['salty',
    'briny',
    'umami',
    'slightly sweet',
    'marine']}],
 1109)

In [235]:
df = pd.DataFrame(all_ing)
df = df.rename(columns={"matched_flavor_descriptors": "generic_flavor_descriptors"})

rag = pd.DataFrame(native_ing)
# df.loc[df['scientific_name'] == 'Palmaria palmata', 'answer'].values
rag

,prompt,scientific_name,answer,matched_flavor_descriptors
0,"What does varech géant,, with the scientific n...",Macrocystis pyrifera,"WeWe need to answer: ""What does varech géant, ...","[seaweed, briny, umami, salty, slightly sweet,..."
1,"What does Dulse, with the scientific name Palm...",Palmaria palmata,"We need to answer: ""What does Dulse, with the ...","[sharp, nutty, hard, salty, briny, umami, spic..."
2,"What does Laver, with the scientific name Porp...",Porphyra perforata,"The task: ""What does Laver, with the scientifi...","[seaweed, salty, popcorn, crisp, umami]"
3,"What does Laver, with the scientific name Porp...",Porphyra abbottae,"salty, umami, briny, mineral, oceanic, slightl...","[salty, umami, briny, mineral, oceanic, slight..."
4,"What does Laver seaweed \n, with the scientifi...",Porphyra torta,"salty, umami, briny, slightly sweet, mineral\n...","[salty, umami, briny, slightly sweet, mineral]"
...,...,...,...,...
163,"What does Great Burdock, with the scientific n...",Arcticum lappa,"mild, nutty, somewhat sweet, starchy, soothing...","[mild, nutty, sweet, starchy, soothing, earthy..."
164,"What does Common Burdock, with the scientific ...",Arcticum minus,"The question: ""What does Common Burdock, with ...","[leaf, raw, salt, the young, tender]"
165,"What does Armoise des champs​, with the scient...",Artemisia campestris,"bitter, aromatic, earthy, herbaceous, slightly...","[bitter, aromatic, earthy, herbaceous, slightl..."
166,"What does Dragon Sagewort , with the scientifi...",Artemisia dracunculus,"warm, faintly aniseed, subtle, peppery, sweet,...","[warm, aniseed, subtle, peppery, sweet, licori..."


In [236]:
# Keep only the RAG cols you want, prefix them with rag_
rag_pref = rag.rename(columns={
    "answer": "rag_answer",
    "matched_flavor_descriptors": "rag_matched_flavor_descriptors",
})

df = df.merge(
    rag_pref[["scientific_name", "rag_answer", "rag_matched_flavor_descriptors"]],
    left_on="common_name",
    right_on="scientific_name",
    how="left"
).rename(columns={"scientific_name_x": "scientific_name"}).drop(columns=["scientific_name_y", "scientific_name"], errors="ignore") 

# Keep Rag value if available
df["matched_flavor_descriptors"] = (
    df["rag_matched_flavor_descriptors"]
    .where(df["rag_matched_flavor_descriptors"].notna(),
           df["generic_flavor_descriptors"])
)

df

,prompt,common_name,answer,generic_flavor_descriptors,rag_answer,rag_matched_flavor_descriptors,matched_flavor_descriptors
0,"What does Macrocystis pyrifera, taste like? Gi...",Macrocystis pyrifera,"salty, briny, umami, slightly sweet, marine\n\...","[salty, briny, umami, slightly sweet, marine]","WeWe need to answer: ""What does varech géant, ...","[seaweed, briny, umami, salty, slightly sweet,...","[seaweed, briny, umami, salty, slightly sweet,..."
1,"What does Palmaria palmata, taste like? Give a...",Palmaria palmata,"salty, briny, slightly sweet, umami\n\n(Note: ...","[salty, briny, slightly sweet, umami]","We need to answer: ""What does Dulse, with the ...","[sharp, nutty, hard, salty, briny, umami, spic...","[sharp, nutty, hard, salty, briny, umami, spic..."
2,"What does Porphyra perforata, taste like? Give...",Porphyra perforata,"The user asks: ""What does Porphyra perforata, ...","[seaweed, umami, salty, briny, slightly sweet,...","The task: ""What does Laver, with the scientifi...","[seaweed, salty, popcorn, crisp, umami]","[seaweed, salty, popcorn, crisp, umami]"
3,"What does Porphyra abbottae, taste like? Give ...",Porphyra abbottae,"salty, briny, umami, slightly sweet, mineral\n...","[salty, briny, umami, slightly sweet, mineral]","salty, umami, briny, mineral, oceanic, slightl...","[salty, umami, briny, mineral, oceanic, slight...","[salty, umami, briny, mineral, oceanic, slight..."
4,"What does Porphyra torta, taste like? Give a c...",Porphyra torta,"salty, briny, umami, slightly sweet, mineral\n...","[salty, briny, umami, slightly sweet, mineral]","salty, umami, briny, slightly sweet, mineral\n...","[salty, umami, briny, slightly sweet, mineral]","[salty, umami, briny, slightly sweet, mineral]"
...,...,...,...,...,...,...,...
1104,"What does Green zucchini, taste like? Give a c...",Green zucchini,"mild, slightly sweet, fresh, grassy, subtly nu...","[mild, slightly sweet, fresh, grassy, subtly n...",NaN,NaN,"[mild, slightly sweet, fresh, grassy, subtly n..."
1105,"What does Yellow zucchini, taste like? Give a ...",Yellow zucchini,"sweet, buttery, mild\n\n(Note: The answer is a...","[sweet, buttery, mild]",NaN,NaN,"[sweet, buttery, mild]"
1106,"What does Saskatoon berry, taste like? Give a ...",Saskatoon berry,"sweet, tart, slightly earthy, berry-like\n\n(N...","[sweet, tart, slightly earthy, berry-like, earth]",NaN,NaN,"[sweet, tart, slightly earthy, berry-like, earth]"
1107,"What does Nanking cherry, taste like? Give a c...",Nanking cherry,"We need to answer: ""What does Nanking cherry, ...","[cherry, tart, sweet, slightly astringent, fru...",NaN,NaN,"[cherry, tart, sweet, slightly astringent, fru..."


# Clean GPT response

In [237]:
merged_df = pd.read_csv('data/merged_ai_descriptors.csv')

merged_df['ai_descriptors'] = merged_df['ai_descriptors'].apply(remove_parentheses_text)
merged_df['ai_descriptors']= merged_df['ai_descriptors'].str.replace(", ", ",")


merged_df = merged_df.merge(
    df[
        ["common_name", "prompt", "answer", "matched_flavor_descriptors"]
    ],
    left_on="Nom scientifique",
    right_on="common_name",
    how="left"
).drop(columns="common_name")

merged_df = merged_df.rename(columns={
    "ai_descriptors": "original_ai_descriptors",
    "matched_flavor_descriptors": "ai_descriptors"
})

merged_df["ai_descriptors"] = merged_df["ai_descriptors"].apply(
    lambda x: ",".join(x) if isinstance(x, list) else x
)

merged_df.head()

,Nom scientifique,original_ai_descriptors,sub_category,db,category,prompt,answer,ai_descriptors
0,Macrocystis pyrifera,"briny,umami,salty,oceanic,mineral,vegetal,slig...",plant,local,plant,"What does Macrocystis pyrifera, taste like? Gi...","salty, briny, umami, slightly sweet, marine\n\...","seaweed,briny,umami,salty,slightly sweet,marin..."
1,Palmaria palmata,"briny,salty,umami,oceanic,slightly sweet,miner...",plant,local,plant,"What does Palmaria palmata, taste like? Give a...","salty, briny, slightly sweet, umami\n\n(Note: ...","sharp,nutty,hard,salty,briny,umami,spicy,wild,..."
2,Porphyra perforata,"briny,umami,salty,mineral,oceanic,slightly swe...",plant,local,plant,"What does Porphyra perforata, taste like? Give...","The user asks: ""What does Porphyra perforata, ...","seaweed,salty,popcorn,crisp,umami"
3,Porphyra abbottae,"briny,umami,slightly sweet,mineral,oceanic,veg...",plant,local,plant,"What does Porphyra abbottae, taste like? Give ...","salty, briny, umami, slightly sweet, mineral\n...","salty,umami,briny,mineral,oceanic,slightly swe..."
4,Porphyra torta,"briny,oceanic,umami,mineral,slightly sweet,veg...",plant,local,plant,"What does Porphyra torta, taste like? Give a c...","salty, briny, umami, slightly sweet, mineral\n...","salty,umami,briny,slightly sweet,mineral"


In [238]:
# Text cleanup
lex = load_lexicon("data/lexicon.xlsx")  # ensures index='desc' and has column 'adj'

merged_df = add_clean_descriptors_column(
    df=merged_df,
    desc_col="ai_descriptors",
    name_col="Nom scientifique",
    lex=lex,
    adjectives=DEFAULT_ADJECTIVES,
    max_len=30,
    out_col="descriptor"
)

merged_df.head()


,Nom scientifique,original_ai_descriptors,sub_category,db,category,prompt,answer,ai_descriptors,descriptor
0,Macrocystis pyrifera,"briny,umami,salty,oceanic,mineral,vegetal,slig...",plant,local,plant,"What does Macrocystis pyrifera, taste like? Gi...","salty, briny, umami, slightly sweet, marine\n\...","seaweed,briny,umami,salty,slightly sweet,marin...","seaweed,brine,umami,salt,sweet,marine,mineral,..."
1,Palmaria palmata,"briny,salty,umami,oceanic,slightly sweet,miner...",plant,local,plant,"What does Palmaria palmata, taste like? Give a...","salty, briny, slightly sweet, umami\n\n(Note: ...","sharp,nutty,hard,salty,briny,umami,spicy,wild,...","sharp,nut,hard,salt,brine,umami,spicy,wild,ses..."
2,Porphyra perforata,"briny,umami,salty,mineral,oceanic,slightly swe...",plant,local,plant,"What does Porphyra perforata, taste like? Give...","The user asks: ""What does Porphyra perforata, ...","seaweed,salty,popcorn,crisp,umami","seaweed,salt,popcorn,crisp,umami"
3,Porphyra abbottae,"briny,umami,slightly sweet,mineral,oceanic,veg...",plant,local,plant,"What does Porphyra abbottae, taste like? Give ...","salty, briny, umami, slightly sweet, mineral\n...","salty,umami,briny,mineral,oceanic,slightly swe...","salt,umami,brine,mineral,ocean,sweet,earth,cri..."
4,Porphyra torta,"briny,oceanic,umami,mineral,slightly sweet,veg...",plant,local,plant,"What does Porphyra torta, taste like? Give a c...","salty, briny, umami, slightly sweet, mineral\n...","salty,umami,briny,slightly sweet,mineral","salt,umami,brine,sweet,mineral"


In [239]:
# Look at the distribution of descriptors
descriptor_counts = (
    merged_df['descriptor']
    .dropna()                       # remove missing values
    .str.split(',')                 # split each cell into a list
    .explode()                      # make one row per descriptor
    .str.strip()                    # remove extra spaces
    .value_counts()                 # count each descriptor
)

print(descriptor_counts.head(20))

descriptor
sweet       874
earth       415
nut         406
butter      324
tang        302
bitter      245
aroma       158
fresh       145
delicate    133
citrus      131
cream       116
salt        103
savory      103
fruit       100
umami        92
spicy        90
floral       90
pepper       87
tart         87
brine        76
Name: count, dtype: int64


In [240]:
merged_df.to_csv('data/merged_ai_descriptors_clean.csv', index=False)

# One Hot Encoding

In [241]:
# Create a copy of the dataframe for this method
df = merged_df.copy()

# The .str.get_dummies() method handles the splitting and encoding in one step.
# We just need to provide the separator character.
flavor_dummies = merged_df['descriptor'].str.get_dummies(sep=',')

# Concatenate the new dummies DataFrame with the original one.
# We drop the original 'genres' column.
df = pd.concat([df[['Nom scientifique']], flavor_dummies], axis=1)
df1 = df.copy(deep = True)

df1.to_csv('data/merged_ai_descriptors_dummies_full.csv', index=False)
df1.head()

,Nom scientifique,acacia,acid,aged,airy,alcohol,allium,allspice,almond,amber,...,watercress,wheat,wholesome,wild,wine,winey,wood,yeast,ylang,zest
0,Macrocystis pyrifera,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Palmaria palmata,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,Porphyra perforata,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Porphyra abbottae,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Porphyra torta,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [242]:
# Calculate the threshold (1% of the number of rows)
threshold = len(df1) * 0.01
threshold = len(df1) * 0.01

# Get the sum of each dummy encoded column (excluding 'Nom scientifique')
dummy_sums = df1.drop(columns=['Nom scientifique']).sum()

# Identify columns where the sum is less than the threshold
columns_to_drop_based_on_threshold = dummy_sums[dummy_sums < threshold].index.tolist()

# Drop these columns from the dataframe
df1 = df1.drop(columns=columns_to_drop_based_on_threshold)

print(f"Dropped {len(columns_to_drop_based_on_threshold)} columns with less than 1% True values.")
print(f"Remaining columns: {df1.shape[1]}")

Dropped 250 columns with less than 1% True values.
Remaining columns: 73


adding categories one hot encoded

In [243]:
# adding food categories
# merged_df.head()

cat_dummies = merged_df['category'].str.get_dummies(sep=',')

## Looks like no correction is best - else it is heavily biased toward category - maybe use man made category or pca to soften the impact of categories
# # correction option 1 correct for the number of columns
# flavor_cols = df1.shape[1] - 1
# cat_cols = cat_dummies.shape[1]
# cat_correction = flavor_cols/cat_cols
# print(f"The number of times more flavor columns than category columns is : {cat_correction}")
# cat_dummies = cat_dummies.multiply(cat_correction)

# # Correction option 2: get average row sum of flavor cols
# df1_numeric = df1.drop(columns=['Nom scientifique'])
# average_row_sum = df1_numeric.sum(axis=1).mean()
# print(f"The average row sum for df1 is: {average_row_sum}")

# cat_dummies = cat_dummies.multiply(average_row_sum )

# Combine the 2 column sets
for col in cat_dummies.columns:
  cat_col = "cat_" + col
  df1[cat_col] = cat_dummies[col]

In [244]:
df1.to_csv('data/merged_ai_descriptors_dummies_filtered.csv', index=False)